In [1]:
import torch
from PIL import Image
import timm
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform
from torchvision import transforms
from typing import Union


class UNI2ExtractorCrops:
    def __init__(self, device: str = "cuda"):
        self.device = torch.device(device if torch.cuda.is_available() else "cpu")

        timm_kwargs = {
            "img_size": 224,
            "patch_size": 14,
            "depth": 24,
            "num_heads": 24,
            "init_values": 1e-5,
            "embed_dim": 1536,
            "mlp_ratio": 2.66667 * 2,
            "num_classes": 0,
            "no_embed_class": True,
            "mlp_layer": timm.layers.SwiGLUPacked,
            "act_layer": torch.nn.SiLU,
            "reg_tokens": 8,
            "dynamic_img_size": True,
        }

        # Load UNI2-h model
        self.model = timm.create_model(
            "hf-hub:MahmoodLab/UNI2-h",
            pretrained=True,
            **timm_kwargs
        ).to(self.device)
        self.model.eval()

        # Base evaluation transform
        config = resolve_data_config(self.model.pretrained_cfg, model=self.model)
        self.eval_transform = create_transform(**config)

        # === FiveCrop ===
        self.fivecrop_transform = transforms.Compose([
            transforms.FiveCrop(256),
            transforms.Lambda(
                lambda crops: torch.stack([self.eval_transform(c) for c in crops])
            )
        ])

        # === TenCrop ===
        self.tencrop_transform = transforms.Compose([
            transforms.TenCrop(256),
            transforms.Lambda(
                lambda crops: torch.stack([self.eval_transform(c) for c in crops])
            )
        ])

    # ------------------------------------------------------------------
    def extract_fivecrop_features(self, image: Union[str, Image.Image]) -> torch.Tensor:
        """
        Extract features from 5 crops.
        Returns tensor of shape: (5, 1536)
        """
        image = self._load_image(image)
        crops_tensor = self.fivecrop_transform(image).to(self.device)

        with torch.no_grad():
            features = self.model(crops_tensor)

        return features.cpu()

    # ------------------------------------------------------------------
    def extract_tencrop_features(self, image: Union[str, Image.Image]) -> torch.Tensor:
        """
        Extract features from 10 crops.
        Returns tensor of shape: (10, 1536)
        """
        image = self._load_image(image)
        crops_tensor = self.tencrop_transform(image).to(self.device)

        with torch.no_grad():
            features = self.model(crops_tensor)

        return features.cpu()

    # ------------------------------------------------------------------
    @staticmethod
    def _load_image(image: Union[str, Image.Image]) -> Image.Image:
        if isinstance(image, str):
            return Image.open(image).convert("RGB")
        elif isinstance(image, Image.Image):
            return image.convert("RGB")
        else:
            raise ValueError(f"Unsupported image type: {type(image)}")


<h1>Testing<h1>

In [2]:
# FiveCrop / TenCrop feature extraction using UNI2H

import glob
import os
from tqdm import tqdm
import torch

# Set paths
patch_root = r"D:\Aamir Gulzar\KSA_project2\dataset\test_patch_data"
save_root = r"D:\Aamir Gulzar\KSA_project2\dataset\Features\UNI2H_tencrop_test"

# Initialize UNI2H extractor
extractor = UNI2ExtractorCrops()

# Loop through all PNG patches
all_patch_paths = glob.glob(os.path.join(patch_root, "*", "*.png"))

for patch_path in tqdm(all_patch_paths, desc="🔄 Extracting UNI2H features"):
    try:
        # --- Choose ONE ---
        # FiveCrop
        # features = extractor.extract_fivecrop_features(patch_path)  # (5, D)

        # TenCrop
        features = extractor.extract_tencrop_features(patch_path)  # (10, D)

        # Preserve slide-wise folder structure
        relative_path = os.path.relpath(patch_path, patch_root)
        slide_folder, patch_filename = os.path.split(relative_path)
        patch_base = os.path.splitext(patch_filename)[0]

        save_dir = os.path.join(save_root, slide_folder)
        os.makedirs(save_dir, exist_ok=True)

        save_path = os.path.join(save_dir, patch_base + ".pt")
        torch.save(features, save_path)

    except Exception as e:
        print(f"❌ Error processing {patch_path}: {e}")

🔄 Extracting UNI2H features:   0%|          | 0/6164 [00:00<?, ?it/s]c:\Users\datainsight\anaconda3\envs\exaonepath\lib\site-packages\timm\layers\attention.py:80: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  x = F.scaled_dot_product_attention(
🔄 Extracting UNI2H features: 100%|██████████| 6164/6164 [15:30<00:00,  6.62it/s]


In [3]:
import os
import glob
import pandas as pd
import torch
from tqdm import tqdm

def run_full_pipeline_UNI2H_crops(
    patch_data_dir: str,
    extractor,  # Instance of UNI2HExtractorCrops
    non_white_csv: str,
    output_root: str,
    tencrop: bool = True
):
    """
    Run UNI2H feature extraction (FiveCrop or TenCrop) on patch dataset.
    Skips white patches and resumes safely if interrupted.
    """

    # Load non-white metadata
    non_white_df = pd.read_csv(non_white_csv)
    valid_patch_names = set(non_white_df['patch_name'].astype(str))

    # Loop over slide directories
    slide_dirs = glob.glob(os.path.join(patch_data_dir, "*"))
    for slide_dir in tqdm(slide_dirs, desc="🧩 Processing slides", unit="slide", position=0):
        slide_name = os.path.basename(slide_dir)
        patch_feat_dir = os.path.join(output_root, slide_name)
        os.makedirs(patch_feat_dir, exist_ok=True)

        # Collect valid non-white patches
        all_patches = glob.glob(os.path.join(slide_dir, "*.png"))
        valid_patch_paths = [
            p for p in all_patches
            if os.path.basename(p) in valid_patch_names
        ]

        if not valid_patch_paths:
            tqdm.write(f"  ❌ No valid non-white patches found for {slide_name}")
            continue

        # Skip already processed patches
        existing_feats = {
            os.path.splitext(f.name)[0]
            for f in os.scandir(patch_feat_dir)
            if f.name.endswith(".pt")
        }

        to_process = [
            p for p in valid_patch_paths
            if os.path.splitext(os.path.basename(p))[0] not in existing_feats
        ]

        if not to_process:
            tqdm.write(
                f"  ⏭️ Skipped: All {len(valid_patch_paths)} patches already processed for {slide_name}"
            )
            continue

        # Patch-level progress bar
        for patch_path in tqdm(
            to_process,
            desc=f"🔄 {slide_name}",
            unit="patch",
            position=1,
            leave=False
        ):
            patch_name = os.path.splitext(os.path.basename(patch_path))[0]
            save_path = os.path.join(patch_feat_dir, patch_name + ".pt")

            try:
                if tencrop:
                # Choose ONE depending on your extractor
                # features = extractor.extract_fivecrop_features(patch_path)  # (5, D)
                    features = extractor.extract_tencrop_features(patch_path)    # (10, D)
                else:
                    features = extractor.extract_fivecrop_features(patch_path)  # (5, D)
                    
                torch.save(features, save_path)

            except Exception as e:
                tqdm.write(f"  ❌ Failed to process patch {patch_path}: {e}")
                continue

        tqdm.write(
            f"  ✅ Done: Processed {len(to_process)} new patches "
            f"(total: {len(valid_patch_paths)}) for {slide_name}"
        )

In [4]:
extractor = UNI2ExtractorCrops()

run_full_pipeline_UNI2H_crops(
    patch_data_dir=r"D:\Aamir Gulzar\KSA_project2\dataset\patch_data",
    extractor=extractor,
    non_white_csv=r"D:\Aamir Gulzar\KSA_project2\dataset\final_merged_without_white.csv",
    output_root=r"D:\Aamir Gulzar\KSA_project2\dataset\Features\UNI2H_tencrop"
)

🧩 Processing slides:   0%|          | 1/417 [00:00<01:10,  5.89slide/s]

  ⏭️ Skipped: All 1878 patches already processed for TCGA-3L-AA1B_nonMSIH


🧩 Processing slides:   0%|          | 2/417 [00:00<01:41,  4.08slide/s]

  ⏭️ Skipped: All 4240 patches already processed for TCGA-4N-A93T_nonMSIH


🧩 Processing slides:   1%|          | 3/417 [00:00<01:54,  3.60slide/s]

...

  ⏭️ Skipped: All 5746 patches already processed for TCGA-SS-A7HO_nonMSIH


🧩 Processing slides: 100%|█████████▉| 416/417 [02:25<00:00,  1.73slide/s]

  ⏭️ Skipped: All 5462 patches already processed for TCGA-T9-A92H_nonMSIH


🧩 Processing slides: 100%|██████████| 417/417 [02:26<00:00,  2.85slide/s]

  ⏭️ Skipped: All 4532 patches already processed for TCGA-WS-AB45_MSIH


In [6]:
import os

output_root = r"D:\Aamir Gulzar\KSA_project2\dataset\Features\UNI2H_tencrop"

total_folders = 0

for dirpath, dirnames, filenames in os.walk(output_root):
    total_folders += len(dirnames)

print(f"Total number of folders (including all subfolders): {total_folders}")

Total number of folders (including all subfolders): 417
